# 第 4 章 · 工具系统：让 Agent 从"会说"到"会做"

> **你已会**：Agent 参数、`instruction`、事件流、`output_key`。
>
> **读完后你能**：① 写一个模型愿意正确调用的工具；② 用 `ToolContext` 读写会话状态；③ 判断什么时候该用 `AgentTool`；④ 用本地 MCP Server 扩展工具集；⑤ 实现长时任务的两阶段恢复；⑥ 从 OpenAPI 规格生成工具。
>
> **本章不碰**：子 Agent 转移（第 5 章）、回调拦截（第 7 章）、长期记忆（第 6 章）。

**怎么读**

- **主线（约 50 分钟）**：§1 事件流 → §2 FunctionTool → §3 ToolContext → §4 AgentTool。
- **深读**：§5 内置工具与 MCP，§6 长时任务，§7 工具确认，§8 OpenAPI。
- 每节先看「先猜」，再运行。

---

## 1. 工具调用在事件流中的样子

第 1 章说过：理解事件流就理解了 ADK 的一半。工具调用会让事件流多出两个环节：

```mermaid
sequenceDiagram
    participant U as 用户
    participant A as Agent(模型)
    participant F as 你的 Python 函数
    U->>A: 北京明天天气怎么样？
    Note over A: 模型判断：需要调用工具
    A-->>U: Event① function_call: get_weather(city="北京")
    A->>F: 框架执行函数
    F-->>A: 返回 {"天气": "晴"}
    A-->>U: Event② function_response: 工具结果
    Note over A: 模型基于结果组织语言
    A-->>U: Event③ 最终文本回复
```

> 📌 **先记住这个分工**：模型只负责"决定调什么、传什么参数"和"解读结果"；函数真正运行在**你的 Python 进程**里，可以访问数据库、内网 API、文件系统。

**先猜：如果模型想调用一个不存在的工具，会发生什么？** 是报错、重试，还是自己编一个答案？学完本章你可以用实验回答。

---

## 2. FunctionTool：函数即工具

ADK 的写法很直接：**把普通 Python 函数放进 `tools` 列表**。框架会自动把它包装成 `FunctionTool`。

一个合格的工具有四个要素：

| 要素 | 来自哪里 | 模型用它做什么 |
|---|---|---|
| 工具名 | 函数名 | 决定"这个工具是干什么的" |
| 工具描述 | **docstring** | 决定"什么时候该调用它" |
| 参数 schema | **类型注解** | 决定"参数怎么填" |
| 返回值 | 最好用 `dict` | 解读结果并继续推理 |

> ⚠️ **docstring 不是写给人看的摆设。** 它是模型唯一能看到的"使用说明书"。描述含糊，是工具被误用或干脆不被调用的头号原因。

下面给 Agent 配两个工具，并把完整事件流打印出来：

In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置 DEEPSEEK_API_KEY"

from google.adk.agents import Agent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP, USER = "adk_ch03", "student"

async def run_verbose(agent, query, session_id="demo"):
    """运行一轮并打印完整事件流（工具调用过程可见）。"""
    ss = InMemorySessionService()
    await ss.create_session(app_name=APP, user_id=USER, session_id=session_id)
    runner = Runner(agent=agent, app_name=APP, session_service=ss)
    msg = types.Content(role="user", parts=[types.Part(text=query)])
    print(f"🧑 {query}\n" + "─" * 55)
    async for ev in runner.run_async(user_id=USER, session_id=session_id, new_message=msg):
        if not (ev.content and ev.content.parts):
            continue
        for p in ev.content.parts:
            if p.function_call:
                print(f"🔧 [调用] {p.function_call.name}({dict(p.function_call.args)})")
            elif p.function_response:
                print(f"📦 [返回] {p.function_response.name} → {p.function_response.response}")
            elif p.text and ev.is_final_response():
                print(f"🤖 [回复] {p.text}")
    print("─" * 55)

# ---------- 定义两个工具 ----------
def get_weather(city: str) -> dict:
    """查询指定城市的实时天气。

    Args:
        city: 中国城市名，例如"北京"、"上海"。

    Returns:
        包含天气状况与温度的字典。
    """
    fake_db = {"北京": {"状况": "晴", "温度": "26°C"}, "上海": {"状况": "小雨", "温度": "22°C"}}
    return fake_db.get(city, {"状况": "未知（演示数据未收录）", "温度": "-"})

def calculate(expression: str) -> dict:
    """计算一个数学表达式并返回结果。支持加减乘除与括号。

    Args:
        expression: 数学表达式字符串，例如 "(3+5)*2"。
    """
    try:
        result = eval(expression, {"__builtins__": {}}, {})  # 演示环境的安全沙箱写法
        return {"result": result}
    except Exception as e:
        return {"error": str(e)}

assistant = Agent(
    name="tool_demo",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是一个得力助手，需要数据时主动调用工具，不要自己编造。",
    description="带天气和计算工具的助手",
    tools=[get_weather, calculate],   # ← 函数直接放进列表
)

await run_verbose(assistant, "上海现在天气如何？如果穿短袖的建议温度是25度以上，(26-22)*3 是多少？")


🧑 上海现在天气如何？如果穿短袖的建议温度是25度以上，(26-22)*3 是多少？
───────────────────────────────────────────────────────


D:\Projects\PythonProjects\agent-frameworks-learning\.venv\Lib\site-packages\google\adk\models\llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


14:35:34 - LiteLLM:WARNING: get_model_cost_map.py:289 - LiteLLM: Failed to fetch remote model cost map from https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json: _ssl.c:983: The handshake operation timed out. Falling back to local backup.


🔧 [调用] get_weather({'city': '上海'})
🔧 [调用] calculate({'expression': '(26-22)*3'})
📦 [返回] get_weather → {'状况': '小雨', '温度': '22°C'}
📦 [返回] calculate → {'result': 12}


🤖 [回复] 结果如下：

**上海天气**
- 状况：小雨
- 温度：22°C

**计算**
- (26-22)×3 = 12

**关于穿短袖**：上海现在是 22°C，低于 25°C 的建议温度，所以不建议穿短袖，加上还在下小雨，建议带伞并穿件外套。
───────────────────────────────────────────────────────


观察上面的输出，三个环节清晰可辨：`🔧 调用` → `📦 返回` → `🤖 回复`。模型还展现了**组合使用**两个工具的能力。

> ⚠️ **一个容易被忽略的工程问题：工具出错时返回什么？**
>
> 对比两种写法：
>
> ```python
> # 写法 A：抛异常
> def get_order(order_id: str) -> dict:
>     return db[order_id]          # KeyError 直接冒出去
>
> # 写法 B：返回结构化错误
> def get_order(order_id: str) -> dict:
>     if order_id not in db:
>         return {"ok": False, "error": "ORDER_NOT_FOUND", "hint": "请确认订单号"}
>     return {"ok": True, "data": db[order_id]}
> ```
>
> 写法 B 让模型**能读懂失败原因并调整下一步**（例如转去询问用户）；写法 A 往往让整轮运行以异常结束。生产工具应统一采用 B，并保留日志用于排查。
>
> 这不是说异常永远不能抛：参数校验失败、权限不足这类"重试也没用"的错误，可以抛；网络抖动这类"可能恢复"的错误，交给重试机制处理。

---

## 3. ToolContext：工具与状态的双向通道

普通函数是"无状态"的。但很多工具需要读写会话上下文，例如"记录用户偏好""统计调用次数"。

只要给函数加一个 `tool_context: ToolContext` 参数，ADK 就会自动注入上下文对象：

| 能力 | 写法 |
|---|---|
| 读状态 | `tool_context.state.get("key")` |
| 写状态 | `tool_context.state["key"] = value`（会产生 state_delta） |
| 会话信息 | `tool_context.session_id` / `tool_context.user_id` |
| 记忆检索 | `tool_context.search_memory(...)`（第 6 章） |

> 📌 **注入参数的识别规则**：参数名和类型注解共同决定它是不是"框架参数"。普通业务参数照常写，模型会填；`tool_context` 由框架填，模型看不到它。

In [2]:
from google.adk.tools import ToolContext

def save_favorite(topic: str, tool_context: ToolContext) -> dict:
    """把用户喜欢的主题记录到会话状态中。

    Args:
        topic: 用户表示喜欢的主题，例如"科幻电影"。
    """
    favorites = tool_context.state.get("favorites", [])
    favorites.append(topic)
    tool_context.state["favorites"] = favorites   # 写入状态（会被持久化到 Session）
    return {"status": "已记录", "当前收藏": favorites}

def list_favorites(tool_context: ToolContext) -> dict:
    """查看当前会话中已记录的所有喜欢的主题。"""
    return {"favorites": tool_context.state.get("favorites", [])}

memory_assistant = Agent(
    name="memory_demo",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是贴心的兴趣管家：用户表达喜欢什么就调用 save_favorite 记录；用户问收藏就调用 list_favorites。",
    description="记录用户兴趣的管家",
    tools=[save_favorite, list_favorites],
)

await run_verbose(memory_assistant, "我特别喜欢科幻电影，对了还有爵士乐！帮我记下来，然后告诉我现在收藏了啥？", session_id="toolctx")


🧑 我特别喜欢科幻电影，对了还有爵士乐！帮我记下来，然后告诉我现在收藏了啥？
───────────────────────────────────────────────────────


🔧 [调用] save_favorite({'topic': '科幻电影'})
🔧 [调用] save_favorite({'topic': '爵士乐'})
📦 [返回] save_favorite → {'status': '已记录', '当前收藏': ['科幻电影', '爵士乐']}
📦 [返回] save_favorite → {'status': '已记录', '当前收藏': ['科幻电影', '爵士乐']}


🔧 [调用] list_favorites({})
📦 [返回] list_favorites → {'favorites': ['科幻电影', '爵士乐']}


🤖 [回复] 都记下来了！你现在的收藏是：

1. 🎬 **科幻电影**
2. 🎷 **爵士乐**

以后想加新爱好的话，随时告诉我～
───────────────────────────────────────────────────────


> 🔍 注意看：两次工具调用**共享了同一份 Session 状态**。这就是 `ToolContext` 的价值——让工具成为状态机的参与者，而不是孤立函数。

---

## 4. AgentTool：把 Agent 变成工具

第 5 章会讲"层级委派"：父 Agent 把任务**转移**给子 Agent，控制权随之交出去。

但有时你要的是另一种关系：**把专业 Agent 当成一个可调用的函数**——调用完，控制权回到主 Agent 手里。这就是 `AgentTool`：

```mermaid
flowchart LR
    subgraph MT["主 Agent（项目经理）"]
        THINK["模型决策"]
    end
    subgraph AT["AgentTool 封装"]
        TRANS["翻译 Agent<br/>（中英互译专家）"]
    end
    THINK -->|"作为工具调用"| TRANS
    TRANS -->|"返回译文"| THINK
    style TRANS fill:#fce8e6,stroke:#ea4335
```

> 💡 **一句话区分**：
>
> - `AgentTool` 是**调用-返回**：主 Agent 始终握着方向盘，适合"借用专业能力"；
> - `sub_agents` 转移是**移交-接管**：任务和上下文一起交出去，适合"这件事归你负责"。
>
> 第 5 章会系统对比，并给出选择清单。

In [3]:
from google.adk.tools import AgentTool

translator = Agent(
    name="translator",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是专业翻译：输入中文则译成英文，输入英文则译成中文。只输出译文，不加解释。",
    description="中英互译专家",
)

pm = Agent(
    name="pm",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="你是项目经理：先用翻译工具把需求译成英文，再基于英文版本写一句推广语。",
    description="会调用翻译工具的项目经理",
    tools=[AgentTool(agent=translator)],   # ← 整个 Agent 变成一个工具
)

await run_verbose(pm, "需求：我们的新产品是一款能自动整理会议纪要的智能音箱。", session_id="agenttool")


🧑 需求：我们的新产品是一款能自动整理会议纪要的智能音箱。
───────────────────────────────────────────────────────


🔧 [调用] translator({'request': '我们的新产品是一款能自动整理会议纪要的智能音箱。'})


📦 [返回] translator → {'result': 'Our new product is a smart speaker that can automatically organize meeting minutes.'}


🤖 [回复] 英文版本：**Our new product is a smart speaker that can automatically organize meeting minutes.**

基于英文版本撰写推广语：

**"Meet your new meeting teammate — a smart speaker that listens, summarizes, and turns every conversation into clean, ready-to-share minutes, so you can focus on the discussion, not the notes."**

中文参考：认识你的新会议搭档——一台会倾听、会总结的智能音箱，把每场对话自动整理成清晰可分享的会议纪要，让你专注于讨论，而不是记笔记。
───────────────────────────────────────────────────────


> 🔍 观察事件流：翻译 Agent 是以"工具调用"的形式出现的，而不是另一个平等对话者。调用结束后，项目经理继续完成推广语。

---

## 5. 内置工具与 MCP

### 5.1 内置工具：先知道为什么用不上

ADK 预置了一些开箱工具，但它们大多依赖 Google 模型或云凭证：

| 内置工具 | 用途 | 在 DeepSeek 下可用？ |
|---|---|---|
| `google_search` | Google 搜索接地 | ❌ 仅 Gemini 系列 |
| `BuiltInCodeExecutor` | 运行模型生成的代码 | ❌ 仅 Gemini 系列 |
| `VertexAiSearchTool` | 企业搜索 | ❌ 需要 Vertex AI 凭证 |

> 📌 **结论很直接**：本教程的 DeepSeek 线路里，**内置工具全部不可用**。你的工具来源只有三类：`FunctionTool`、`McpToolset`、第三方工具桥接。这不是缺陷——自定义函数工具本来就是实际项目里用得最多的一类。

### 5.2 McpToolset：接入外部工具集

[MCP（Model Context Protocol）](https://modelcontextprotocol.io) 是一个工具开放协议。它解决的是：**别人已经写好的工具，你不需要再复制一遍。**

ADK 通过 `McpToolset` 连接一个 MCP Server。当前版本的正确导入方式是：

```python
import sys

from google.adk.tools import McpToolset
from google.adk.tools.mcp_tool import StdioConnectionParams
from mcp import StdioServerParameters
```

> ⚠️ 2.6.3 中 `MCPToolset` 已标记为 deprecated，新代码统一使用 `McpToolset`。

仓库里带了一个**完全本地、不联网**的最小示例：[mcp_example/local_server.py](mcp_example/local_server.py)。它只暴露一个查询订单状态的工具：

```python
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("local-tools")

_ORDERS = {"A100": "已发货", "B200": "已签收"}

@mcp.tool()
def order_status(order_id: str) -> dict:
    """查询订单状态。"""
    if order_id not in _ORDERS:
        return {"ok": False, "error": "ORDER_NOT_FOUND", "order_id": order_id}
    return {"ok": True, "order_id": order_id, "status": _ORDERS[order_id]}

if __name__ == "__main__":
    mcp.run()
```

接入方式：

```python
mcp_tools = McpToolset(
    connection_params=StdioConnectionParams(
        server_params=StdioServerParameters(
            command=sys.executable,          # 与当前解释器一致，避免 PATH 问题
            args=["mcp_example/local_server.py"],
        ),
        timeout=10,
    )
)

agent = Agent(
    name="order_agent",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="查询订单状态时调用 order_status 工具。",
    tools=[mcp_tools],
)
```

> ⚠️ **两个前提**：① 需要安装 `mcp` 依赖（项目已加入 `pyproject.toml`）；② 路径要相对于你启动 Jupyter 的目录。路径不对时，最典型的报错是"进程启动失败"，而不是"工具找不到"。用 `sys.executable` 可以避免解释器不一致的问题。

> 📌 **MCP 的边界**：它让接入变便宜，也让**权限问题变重要**——一个 MCP Server 能访问什么，模型就间接能访问什么。练习 5 会让你做一次最小安全评审。

### 5.3 第三方工具桥接

ADK 还能包装其他框架的工具：

- **`LangchainTool`**：包装 LangChain 工具；
- **`CrewaiTool`**：包装 CrewAI 工具。

本教程的主线不依赖它们。知道有这条路即可：**当你想复用的工具已有现成实现时，先找桥接，再考虑重写。**

---

## 6. 长时任务：提交、暂停、回填

有些动作一跑就是几分钟：视频渲染、报表生成、外部审批。

`LongRunningFunctionTool` 只解决**等待**问题，它定义的是一个**两阶段协议**：

```text
阶段一：模型调用工具 → 工具返回 pending + task_id → 本轮运行结束
阶段二：任务真正完成后 → 客户端回填 function_response（同一个 call id）
        → 模型拿到结果，继续生成最终回复
```

**先猜：阶段一会不会直接给出最终答案？** 运行下面的代码，观察事件里的 `long_running_tool_ids`。

> 📌 **框架只管"挂起和回填"，不管任务本身。** 任务 ID 存哪、进度怎么查、重复提交怎么去重、进程重启怎么恢复，仍然是你自己的工程问题。


In [4]:
from google.adk.tools import LongRunningFunctionTool

TASKS: dict[str, dict] = {}

def submit_report(topic: str) -> dict:
    """提交一个耗时的报告生成任务，立即返回任务 ID。

    Args:
        topic: 报告主题。
    """
    task_id = f"T-{len(TASKS) + 1}"
    TASKS[task_id] = {"topic": topic, "status": "pending"}
    return {"task_id": task_id, "status": "pending"}

long_agent = Agent(
    name="report_agent",
    model=LiteLlm(model="deepseek/deepseek-chat"),
    instruction="用户要求生成报告时，必须调用 submit_report 工具，不要假装已经完成。",
    description="提交长时报告任务的助手",
    tools=[LongRunningFunctionTool(submit_report)],
)

long_service = InMemorySessionService()
await long_service.create_session(app_name=APP, user_id=USER, session_id="long_run")
long_runner = Runner(agent=long_agent, app_name=APP, session_service=long_service)

message = types.Content(role="user", parts=[types.Part(text="帮我生成一份 ADK 学习报告。")])
call_id, long_running_ids = None, set()
async for event in long_runner.run_async(
    user_id=USER, session_id="long_run", new_message=message
):
    long_running_ids |= set(event.long_running_tool_ids or [])
    for part in (event.content.parts if event.content else []):
        if part.function_call:
            call_id = part.function_call.id
        if part.function_response:
            print("阶段一：工具返回", part.function_response.response)

assert call_id and call_id in long_running_ids, "没有识别出长时任务调用"
assert TASKS["T-1"]["status"] == "pending"
print("阶段一：任务已挂起，等待回填：", TASKS)

# 模拟外部任务完成：用同一个 call id 回填 function_response
TASKS["T-1"]["status"] = "done"
resume = types.Content(
    role="user",
    parts=[
        types.Part(
            function_response=types.FunctionResponse(
                id=call_id,
                name="submit_report",
                response={"task_id": "T-1", "status": "done", "url": "report.md"},
            )
        )
    ],
)

final_text = ""
async for event in long_runner.run_async(
    user_id=USER, session_id="long_run", new_message=resume
):
    if event.is_final_response() and event.content:
        final_text += "".join(p.text or "" for p in event.content.parts)

assert "T-1" in final_text, "回填后模型没有使用任务结果"
print("阶段二：", final_text[:160])
print("✅ 两阶段协议完成：提交 → pending → 回填 → 最终回复")

阶段一：工具返回 {'task_id': 'T-1', 'status': 'pending'}


阶段一：任务已挂起，等待回填： {'T-1': {'topic': 'ADK 学习报告', 'status': 'pending'}}


阶段二： 已为你提交 ADK 学习报告任务，任务已完成。

- 任务 ID：T-1
- 状态：done
- 报告地址：report.md

你可以直接查看 report.md 获取内容。如果需要调整报告的重点（例如更偏向架构原理、代码实践或对比分析），告诉我，我可以再提交一版。
✅ 两阶段协议完成：提交 → pending → 回填 → 最终回复



---

## 7. 工具确认与人工审批

高风险工具（退款、删库、发邮件）不能只靠提示词约束。ADK 在 `FunctionTool` 上提供了框架级的确认开关：

```python
confirmed_tool = FunctionTool(create_refund, require_confirmation=True)
```

它会在执行前发出一个确认请求，等客户端回传"已确认"后才真正执行。这个能力需要 UI 或客户端配合，本教程不展开。

本地更常用的做法是**审批令牌**：工具参数里带审批码，回调先行拦截，工具本体再复核一次。

> ⚠️ **回调是策略执行点，不是唯一安全边界。** 工具本体必须自己再检查一次，否则任何绕过回调的调用路径都会成为漏洞。


In [5]:
from google.adk.tools import FunctionTool

REFUND_LEDGER: dict[str, int] = {}

def create_refund(order_id: str, amount: int, approval_code: str = "") -> dict:
    """执行退款。工具本体也会复核审批，不能只依赖回调。

    Args:
        order_id: 订单号。
        amount: 退款金额。
        approval_code: 人工审批码。
    """
    if approval_code != "APPROVED":
        return {"ok": False, "error": "APPROVAL_REQUIRED"}
    if order_id in REFUND_LEDGER:
        return {"ok": True, "duplicate": True}
    REFUND_LEDGER[order_id] = amount
    return {"ok": True, "duplicate": False}

def approval_gate(tool, args, tool_context):
    """before_tool 回调：没有审批码就短路，工具不会执行。"""
    if not args.get("approval_code"):
        return {"ok": False, "error": "APPROVAL_REQUIRED", "blocked": True}
    return None

def attempt_refund(args: dict) -> dict:
    decision = approval_gate(None, args, None)
    if decision is not None:
        return decision
    return create_refund(**args)

assert attempt_refund({"order_id": "A100", "amount": 299})["error"] == "APPROVAL_REQUIRED"
assert REFUND_LEDGER == {}, "被拦截时不能产生副作用"

assert attempt_refund(
    {"order_id": "A100", "amount": 299, "approval_code": "APPROVED"}
)["ok"]
assert len(REFUND_LEDGER) == 1
print("✅ 审批缺失时零副作用；带审批码时只执行一次")

# 框架级确认开关（需要 UI/客户端回传确认，本教程不运行）：
# confirmed_tool = FunctionTool(create_refund, require_confirmation=True)

✅ 审批缺失时零副作用；带审批码时只执行一次



---

## 8. 用 OpenAPI 规格生成工具

如果目标服务已经有 OpenAPI 文档，你不需要手写每个函数。`OpenAPIToolset` 会读取规格并生成工具：

```python
from google.adk.tools.openapi_tool import OpenAPIToolset

toolset = OpenAPIToolset(spec_dict=spec)
agent = Agent(..., tools=[toolset])
```

仓库里放了一份最小的本地规格：[openapi_example/orders.json](openapi_example/orders.json)。下面只做**工具发现**，不发起真实请求：

**先猜：规格里只有一个 `get_order` 操作，生成出来的工具会叫什么名字？**


In [6]:
import json
from pathlib import Path

from google.adk.tools.openapi_tool import OpenAPIToolset

root = Path.cwd()
if not (root / "openapi_example").exists():
    root = root / "google-adk"

spec = json.loads((root / "openapi_example" / "orders.json").read_text(encoding="utf-8"))
openapi_toolset = OpenAPIToolset(spec_dict=spec)
openapi_tools = await openapi_toolset.get_tools()

assert [t.name for t in openapi_tools] == ["get_order"]
print("发现的工具：", [t.name for t in openapi_tools])
print("工具描述：", openapi_tools[0].description)
print("✅ 规格加载成功；真实请求还需要网络、鉴权和超时策略")

发现的工具： ['get_order']
工具描述： 按订单号查询状态。本示例只用于演示工具发现，不发起真实请求。
✅ 规格加载成功；真实请求还需要网络、鉴权和超时策略



> ⚠️ **工具发现不等于工具可用。** OpenAPI 只描述接口；网络可达性、鉴权、超时、错误码映射仍然要你自己保证。

---

<details>
<summary><b>深入：与 LangChain / LangGraph 对照</b></summary>

| ADK | LangChain / LangGraph | 差异 |
|---|---|---|
| 函数直接放入 `tools=[...]` | `@tool` 装饰器或直接传函数 | 体验趋同 |
| docstring 即工具描述 | docstring 或 `@tool(description=...)` | 一致 |
| `ToolContext` 注入 | `ToolRuntime` / `InjectedState` | 思想相同：工具访问图状态 |
| `AgentTool` | 子图封装为工具 / supervisor 模式 | LangGraph 更自由，也更手动 |
| `McpToolset` | `langchain-mcp-adapters` | 都拥抱 MCP |
| `OpenAPIToolset` | 手动包装 API / 自定义工具 | ADK 内建规格解析 |

</details>

---

## 📌 用五句话复述本章

1. **工具 = 函数 + docstring + 类型注解**，docstring 是模型唯一的使用说明书。
2. **模型决定调什么，你的进程负责执行**；工具出错时优先返回结构化错误。
3. **`ToolContext` 让工具读写 Session 状态**；`AgentTool` 是调用-返回，`sub_agents` 是移交-接管。
4. **长时任务是两阶段协议**：提交返回 pending，完成后用同一个 call id 回填。
5. **DeepSeek 下没有可用的内置工具**；补充工具靠 FunctionTool、MCP、OpenAPI。审批必须在工具本体再复核一次。

> ➡️ 下一章：[05-多智能体与工作流](05-多智能体与工作流.ipynb) —— ADK 最精彩的部分。

---

## 🧪 本章练习

### 1. 设计一个可靠的 FunctionTool（基础）

实现"查询订单状态"工具。

- [ ] 输入订单号，返回统一的成功/失败结构
- [ ] 覆盖订单不存在、参数格式错误、下游超时三种情况
- [ ] docstring 写清何时调用、参数怎么填
- [ ] 用测试证明异常不会以未处理错误泄漏给 Agent

### 2. 用 ToolContext 管理购物车（进阶）

- [ ] 实现添加、删除、查看三个工具
- [ ] 通过 `ToolContext` 读写 Session State
- [ ] 支持数量累加、库存不足提示、总价计算
- [ ] 连续对话后检查状态变化
- [ ] 验证不同 Session 的购物车互不影响

### 3. AgentTool 与普通工具的职责划分（综合）

构建"市场简报助手"。

- [ ] 主 Agent 用普通函数工具取得结构化行情数据
- [ ] 把"解释数据并识别风险"交给一个 `AgentTool`
- [ ] 最终输出同时包含原始数据摘要、分析结论、风险声明
- [ ] 说明：为什么取数适合确定性工具、分析适合 AgentTool
- [ ] 断言"先取数、后分析"的调用顺序

### 4. 长时任务的两阶段恢复（工程）

为"生成季度报告"实现长时任务。

- [ ] 提交后返回 task_id，并记录 pending 状态
- [ ] 用同一个 call id 回填完成结果
- [ ] 覆盖完成、失败、取消三条路径
- [ ] 设计重复提交的去重规则
- [ ] 说明进程重启后如何继续

### 5. MCP 接入前的安全评审（开放）

运行仓库里的 `mcp_example/local_server.py`，然后回答：

- [ ] 它能访问哪些数据、执行哪些动作
- [ ] 提示注入和越权风险分别在哪
- [ ] 凭证边界、超时、限流怎么设
- [ ] 应该直接暴露给模型、包一层受控工具，还是仅供人工调用
- [ ] 写出你的结论和理由

### 6. 从 OpenAPI 到工具（综合）

用 `openapi_example/orders.json` 或一份你自己的规格：

- [ ] 生成工具并列出名称与描述
- [ ] 检查参数 schema 是否与规格一致
- [ ] 说明鉴权信息应该放在哪里，不能放在哪里
- [ ] 为下游超时和错误码设计统一映射